# Gene prioritization by link probabilities
---

In [43]:
import os
import pandas as pd
import pickle
from CADA.triples_gene_hpo import triples_gene_hpo
import networkx as nx
import numpy as np
from CADA.gae.preprocessing import mask_test_edges
from gensim.models import Word2Vec
from CADA.paths import DATA_DIRECTORY,MODEL_DIRECTORY

In [44]:
# Collect only gene-hpo term edges
gene_hpo_triples = triples_gene_hpo()
gene_hpo_edges = []
for triple in gene_hpo_triples:
    gene_hpo_edges.append((triple[0], triple[2]))
g_gene_hpo_edges = nx.Graph()
g_gene_hpo_edges.add_edges_from(gene_hpo_edges)
adj_sparse_gene_hpo_edges = nx.to_scipy_sparse_matrix(g_gene_hpo_edges)

# record gene nodes
nodes = list(g_gene_hpo_edges.nodes())
nodes_gene = []
for node in nodes:
    if node.startswith("Entrez:"):
        nodes_gene.append(node)

preprocessing...
generating test/val sets...
creating false test edges...
creating false val edges...
creating false train edges...
final checks for disjointness...
creating adj_train...
Done with train-test split!



In [46]:
# Inspect train/test split
print(f'The number of gene-hpo term edges: {len(g_gene_hpo_edges.edges())}')
print(f'Training edges (positive): {len(train_edges)}')
print(f'Training edges (negative): {len(train_edges_false)}')
print(f'Validation edges (positive): {len(val_edges)}')
print(f'Validation edges (negative): {len(val_edges_false)}')
print(f'Test edges (positive): {len(test_edges)}')
print(f'Test edges (negative): {len(test_edges_false)}')

The number of gene-hpo term edges: 169281
Training edges (positive): 169281
Training edges (negative): 169281
Validation edges (positive): 0
Validation edges (negative): 0
Test edges (positive): 0
Test edges (negative): 0


In [47]:
# get embeddings and create node embeddings matrix(rows = nodes, columns = embedding features)
model = Word2Vec.load('node2vec.model')
emb_mappings = model.wv
emb_list = []
for node in g_gene_hpo_edges.nodes():
    node_emb = emb_mappings[node]
    emb_list.append(node_emb)
emb_matrix = np.vstack(emb_list)

In [48]:
# Generate bootstrapped edge embeddings (as is done in node2vec paper)
# Edge embedding for (v1, v2) = hadamard product of node embeddings for v1, v2
def get_edge_embeddings(edge_list):
    embs = []
    for edge in edge_list:
        node1 = edge[0]
        node2 = edge[1]
        emb1 = emb_matrix[node1]
        emb2 = emb_matrix[node2]
        edge_emb = np.multiply(emb1, emb2)
        embs.append(edge_emb)
    embs = np.array(embs)
    return embs

In [49]:
# Train-set edge embeddings, labels: 1 = real edge, 0 = false edge
pos_train_edge_embs = get_edge_embeddings(train_edges)
neg_train_edge_embs = get_edge_embeddings(train_edges_false)
train_edge_embs = np.concatenate([pos_train_edge_embs, neg_train_edge_embs])
train_edge_labels = np.concatenate([np.ones(len(train_edges)), np.zeros(len(train_edges_false))])

In [50]:
# Train logistic regression classifier on train-set edge embeddings
from sklearn.linear_model import LogisticRegression
edge_classifier = LogisticRegression(random_state=0)
edge_classifier.fit(train_edge_embs, train_edge_labels)

LogisticRegression(C=1.0, class_weight=None, dual=False, fit_intercept=True,
                   intercept_scaling=1, l1_ratio=None, max_iter=100,
                   multi_class='auto', n_jobs=None, penalty='l2',
                   random_state=0, solver='lbfgs', tol=0.0001, verbose=0,
                   warm_start=False)

In [51]:
#count the number of training patients with a specific mutation for each mutation gene
train = os.path.join(os.getcwd(), 'patient_training.tsv')
if os.path.exists(train):
    train_pd = pd.read_csv(train, header=None, sep='\t', names=['patient_id', 'omim_id', 'gene_id', 'features', 'submitter',
                                    'from_file', 'no_features'])
    gene_counts = train_pd['gene_id'].value_counts().to_dict() 
else:
    gene_counts = {}

In [52]:
with open(os.path.join(DATA_DIRECTORY, 'processed', 'ids', 'gene_id_name.dict'), 'rb') as handle:
    gene_id_name = pickle.load(handle)

with open(os.path.join(DATA_DIRECTORY, 'processed', 'ids', 'hpo_id_name.dict'), 'rb') as handle:
    hpo_id_name = pickle.load(handle)

In [54]:
## Prioritizing genes for each test patient based on their hpo terms
with open('patient_testing.tsv', 'r') as t_file:
    content = t_file.read().splitlines()
    content = [x.split('\t') for x in content]
    evaluation_save = list()
    evaluation_vis = list()
    for line in content:
        prioritized_genelist = []
        patient_id = line[0]
        gene_id = line[2]
        # Get the number training patients with same mutation gene
        if gene_id in gene_counts:
            no_patients = gene_counts[gene_id]
        else:
            no_patients = 0
        
        # prioritize genes for each patients by taking the max function of probalities to patient's hpo terms
        features = line[3].split(',')
        table = {}
        for gene in nodes_gene:
            edge_embs = []
            gene_emb = model.wv[gene]
            for feature in features:
                feature_emb = model.wv[feature]
                edge_emb = np.multiply(feature_emb, gene_emb)
                edge_embs.append(edge_emb)
            prob = sum((edge_classifier.predict_proba(edge_embs)[:, 1]).tolist())
            table[gene] = prob
        prioritized_genes = pd.DataFrame.from_dict(table, orient='index', columns=['score']).sort_values(by='score', ascending=False).index.values.tolist()
        # get the rank
        if gene_id in prioritized_genes:
            rank = prioritized_genes.index(gene_id) + 1
        else:
            rank = 'NA'
        evaluation_save.append([patient_id, gene_id, no_patients,','.join(features), prioritized_genes[:30], rank])
        evaluation_vis.append([patient_id, gene_id_name[gene_id], no_patients,
                                   ','.join([hpo_id_name[feature] for feature in features]), ','.join([gene_id_name[gene_id] for gene_id in prioritized_genes[:30]]), rank])
    # save the result
    saveframe = pd.DataFrame(evaluation_save, columns = ['patient_id', 'gene_id','no_patients', 'features', 'result', 'rank'])
    saveframe.to_csv('evaluation.tsv', sep='\t', index=None)
    visframe = pd.DataFrame(evaluation_vis, columns=['patient_id', 'gene', 'no_patients', 'features', 'result', 'rank'])
    visframe.to_excel('evaluation.xlsx', index=None) 
                
                
                

In [55]:
ranks = saveframe['rank'].tolist()
count1 = count5 = count10 = count50 = count100 = count1000 = 0
counts = len(ranks)
for rank in ranks:
    if isinstance(rank, int):
        if rank == 1:
            count1 += 1
        if rank <= 5:
            count5 += 1
        if rank <= 10:
            count10 += 1
        if rank <= 50:
            count50 += 1
        if rank <= 100:
            count100 += 1
        if rank <= 1000:
            count1000 += 1
top1 = round(count1/counts, 2)
top5 = round(count5/counts, 2)
top10 = round(count10/counts, 2)
top50 = round(count50/counts, 2)
top100 = round(count100/counts, 2)
top1000 = round(count1000/counts, 2)

print(f'top1: {top1}')
print(f'top5: {top5}')
print(f'top10: {top10}')
print(f'top50: {top50}')
print(f'top100: {top100}')
print(f'top1000: {top1000}')

top1: 0.04
top5: 0.13
top10: 0.19
top50: 0.39
top100: 0.5
top1000: 0.87
